In [4]:
# ==========================================
# 1. DIRETIVAS DE AMBIENTE (JUPYTER)
# ==========================================
# Habilita a renderização inline de gráficos para análise visual
%matplotlib inline 

# Recarrega dinamicamente os módulos externos (diretório src/) após modificações
%load_ext autoreload
%autoreload 2

# ==========================================
# 2. BIBLIOTECAS STANDARD E MANIPULAÇÃO DE DADOS
# ==========================================
import os
import time
import numpy as np
import pandas as pd

# ==========================================
# 3. BIBLIOTECAS DE VISUALIZAÇÃO
# ==========================================
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração global de estilo para consistência visual
sns.set_theme(style="whitegrid")

# ==========================================
# 4. MÉTODOS DE AVALIAÇÃO E PREPARAÇÃO DE DADOS
# ==========================================
# Nota: O uso do scikit-learn é restrito à avaliação e preparação de dados, 
# respeitando a norma de implementar o algoritmo de classificação "do zero"[cite: 16].
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    confusion_matrix, 
    classification_report
)

# ==========================================
# 5. ESTIMADORES CUSTOMIZADOS (Implementação Própria)
# ==========================================
from src.logistic_regression import LogisticRegression

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
# =====================================================================
# ROTINA DE BENCHMARKING (ESTUDO EMPÍRICO - FASE 1)
# =====================================================================

def evaluate_baseline_models(dataset_dir: str) -> pd.DataFrame:
    """
    Itera sobre todos os ficheiros CSV num diretório específico, treina o modelo 
    de Regressão Logística base e compila um DataFrame com metadados e métricas de desempenho.

    Parâmetros:
    ----------
    dataset_dir : str
        Caminho relativo ou absoluto para o diretório contendo os ficheiros de benchmark.

    Retornos:
    -------
    pd.DataFrame
        DataFrame contendo as características estruturais de cada dataset e as 
        respetivas métricas de classificação obtidas pelo modelo base.
    """
    results_list = []
    
    # Validação do diretório
    if not os.path.exists(dataset_dir):
        print(f"[{time.strftime('%H:%M:%S')}] [Erro] Diretório não encontrado: {dataset_dir}")
        return pd.DataFrame()

    csv_files = [f for f in os.listdir(dataset_dir) if f.endswith('.csv')]
    
    if not csv_files:
        print(f"[{time.strftime('%H:%M:%S')}] [Aviso] Nenhum ficheiro CSV localizado em: {dataset_dir}")
        return pd.DataFrame()

    for file_name in csv_files:
        file_path = os.path.join(dataset_dir, file_name)
        
        # 1. Carregamento de dados e extração de metadados
        df = pd.read_csv(file_path)
        n_samples, n_features_total = df.shape
        n_missing = df.isnull().sum().sum()
        
        # Assume-se que a variável dependente (target) reside na última coluna
        X = df.iloc[:, :-1].values
        y = df.iloc[:, -1].values
        
        n_classes = len(np.unique(y))
        
        # Estrutura de dados para armazenamento do registo atual
        record = {
            'Dataset': file_name,
            'N_Samples': n_samples,
            'N_Features': n_features_total - 1,
            'Missing_Values': n_missing,
            'Target_Classes': n_classes,
            'Execution_Status': 'Pending',
            'Accuracy': np.nan,
            'Precision_Macro': np.nan,
            'Recall_Macro': np.nan,
            'F1_Macro': np.nan,
            'Training_Time_sec': np.nan
        }
        
        # 2. Imputação simples de dados ausentes (necessário para a execução da baseline)
        if n_missing > 0:
            df = df.fillna(df.mean(numeric_only=True))
            X = df.iloc[:, :-1].values
            y = df.iloc[:, -1].values
            
        # 3. Partição dos dados (Holdout method: 80% treino, 20% teste com estratificação)
        try:
            X_train, X_test, y_train, y_test = train_test_split(
                X, y, test_size=0.2, random_state=42, stratify=y
            )
        except Exception as e:
            record['Execution_Status'] = f"Data Split Error: {str(e)}"
            results_list.append(record)
            continue
            
        # 4. Inicialização, treino e avaliação do modelo
        model = LogisticRegression(lr=0.01, max_iters=1000)
        
        start_time = time.time()
        try:
            # Otimização dos parâmetros
            model.fit(X_train, y_train)
            training_time = time.time() - start_time
            
            # Inferência no conjunto de teste
            y_pred = model.predict(X_test)
            
            # Cálculo de métricas (average='macro' para lidar com eventuais assimetrias)
            record['Accuracy'] = accuracy_score(y_test, y_pred)
            record['Precision_Macro'] = precision_score(y_test, y_pred, average='macro', zero_division=0)
            record['Recall_Macro'] = recall_score(y_test, y_pred, average='macro', zero_division=0)
            record['F1_Macro'] = f1_score(y_test, y_pred, average='macro', zero_division=0)
            
            record['Training_Time_sec'] = round(training_time, 4)
            record['Execution_Status'] = 'Success'
            
        except Exception as e:
            # Captura de falhas matemáticas (ex: classes > 2 na regressão logística base)
            record['Execution_Status'] = f"Training/Inference Error: {str(e)}"
            
        results_list.append(record)

    return pd.DataFrame(results_list)

# =====================================================================
# EXECUÇÃO DO PIPELINE PARA OS 3 GRUPOS DE DATASETS
# =====================================================================

# Mapeamento dos diretórios correspondentes aos desafios da Fase 2 (Atualizado com os nomes exatos das pastas)
benchmark_directories = {
    "Grupo1_Noise": "datasets/noise_outliers/",
    "Grupo2_Imbalance": "datasets/class_imbalance/",
    "Grupo3_Multiclass": "datasets/multiclass_classification/"
}

# Estrutura para reter os DataFrames na memória da sessão atual
compiled_results = {}

for group_name, dir_path in benchmark_directories.items():
    print(f"[{time.strftime('%H:%M:%S')}] A iniciar processo de benchmarking para o subset: {group_name}...")
    
    # Avaliação do modelo e compilação de resultados
    df_results = evaluate_baseline_models(dir_path)
    
    if not df_results.empty:
        compiled_results[group_name] = df_results
        
        # Exportação dos resultados consolidados
        output_filename = f"evaluation_baseline_{group_name.lower()}.csv"
        df_results.to_csv(output_filename, index=False)
        print(f"[{time.strftime('%H:%M:%S')}] -> Concluído. Resultados guardados em: {output_filename}\n")
    else:
        print(f"[{time.strftime('%H:%M:%S')}] -> Nenhuma operação realizada para: {group_name} (Verificar diretório).\n")

[15:40:37] A iniciar processo de benchmarking para o subset: Grupo1_Noise...


c:\Users\guilh\Anaconda3\Lib\site-packages\autograd\tracer.py:54: RuntimeWarning: overflow encountered in cosh
  return f_raw(*args, **kwargs)
c:\Users\guilh\Anaconda3\Lib\site-packages\autograd\numpy\numpy_vjps.py:175: RuntimeWarning: overflow encountered in square
  defvjp(anp.tanh, lambda ans, x: lambda g: g / anp.cosh(x) ** 2)


[15:40:38] -> Concluído. Resultados guardados em: evaluation_baseline_grupo1_noise.csv

[15:40:38] A iniciar processo de benchmarking para o subset: Grupo2_Imbalance...


c:\Users\guilh\Anaconda3\Lib\site-packages\autograd\tracer.py:54: RuntimeWarning: overflow encountered in cosh
  return f_raw(*args, **kwargs)
c:\Users\guilh\Anaconda3\Lib\site-packages\autograd\numpy\numpy_vjps.py:175: RuntimeWarning: overflow encountered in square
  defvjp(anp.tanh, lambda ans, x: lambda g: g / anp.cosh(x) ** 2)
c:\Users\guilh\Anaconda3\Lib\site-packages\autograd\tracer.py:54: RuntimeWarning: overflow encountered in cosh
  return f_raw(*args, **kwargs)
c:\Users\guilh\Anaconda3\Lib\site-packages\autograd\numpy\numpy_vjps.py:175: RuntimeWarning: overflow encountered in square
  defvjp(anp.tanh, lambda ans, x: lambda g: g / anp.cosh(x) ** 2)
c:\Users\guilh\Anaconda3\Lib\site-packages\autograd\tracer.py:54: RuntimeWarning: overflow encountered in cosh
  return f_raw(*args, **kwargs)
c:\Users\guilh\Anaconda3\Lib\site-packages\autograd\numpy\numpy_vjps.py:175: RuntimeWarning: overflow encountered in square
  defvjp(anp.tanh, lambda ans, x: lambda g: g / anp.cosh(x) ** 2)
c

[15:40:41] -> Concluído. Resultados guardados em: evaluation_baseline_grupo2_imbalance.csv

[15:40:41] A iniciar processo de benchmarking para o subset: Grupo3_Multiclass...


c:\Users\guilh\Anaconda3\Lib\site-packages\autograd\tracer.py:54: RuntimeWarning: overflow encountered in cosh
  return f_raw(*args, **kwargs)
c:\Users\guilh\Anaconda3\Lib\site-packages\autograd\tracer.py:54: RuntimeWarning: overflow encountered in cosh
  return f_raw(*args, **kwargs)
c:\Users\guilh\Anaconda3\Lib\site-packages\autograd\numpy\numpy_vjps.py:175: RuntimeWarning: overflow encountered in square
  defvjp(anp.tanh, lambda ans, x: lambda g: g / anp.cosh(x) ** 2)
c:\Users\guilh\Anaconda3\Lib\site-packages\autograd\numpy\numpy_vjps.py:175: RuntimeWarning: overflow encountered in square
  defvjp(anp.tanh, lambda ans, x: lambda g: g / anp.cosh(x) ** 2)
c:\Users\guilh\Anaconda3\Lib\site-packages\autograd\tracer.py:54: RuntimeWarning: overflow encountered in cosh
  return f_raw(*args, **kwargs)
c:\Users\guilh\Anaconda3\Lib\site-packages\autograd\tracer.py:54: RuntimeWarning: overflow encountered in cosh
  return f_raw(*args, **kwargs)
c:\Users\guilh\Anaconda3\Lib\site-packages\autog

[15:40:42] -> Concluído. Resultados guardados em: evaluation_baseline_grupo3_multiclass.csv

